# Learned critics and frozen text rewards

A critic estimates what has not yet happened. That is useful—and dangerous when its guesses become the targets that train it. This lesson connects terminal reward, EOS/cap masks, TD/GAE, separate actor/critic gradients and actual fitted text rewards.

Predict → inspect tensors → run a small neural update → perturb masks → confront all fixed-budget outcomes. The measured finite grammar is not a pretrained PPO system or evidence of general language skill. [Chapter 12](../../book/chapters/12-language-generation-as-a-policy.md) · [specification](../../experiments/specs/2026-10-04-critic-policy.md) · [report](../../experiments/reports/2026-10-04-critic-policy.md)

In [ ]:
from pathlib import Path
import sys,json
root=Path.cwd().resolve()
while not (root/"src"/"dongxi_llms").is_dir():
    if root.parent==root: raise RuntimeError("Open within Dongxi_LLMs")
    root=root.parent
sys.path.insert(0,str(root/"src"))
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from dongxi_llms.critic_policy_lab import (
    gae_targets,load_protocol,run_arm,state_hash,terminal_paths
)
from dongxi_llms.char_reward_lab import load_char_reward
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","font.family":"DejaVu Sans",
                     "font.size":10,"axes.spines.top":False,"axes.spines.right":False})
report=json.loads((root/"experiments/reports/2026-10-04-critic-policy-verification.json").read_text())
result=report["results"];protocol=load_protocol();runs=result["runs"]
colors={"oracle":"#356a9b","learned":"#b97139","noisy":"#3d8268","confounded":"#775c91"}
assert len(runs)==12 and not result["failures"]
print("Actual measured run:",report["environment"],report["seconds"])
print("Finite terminal paths:",len(terminal_paths()),"fixed seeds:",protocol["actor_seeds"])


## Exercise 1 — which model should a gradient update?

Predict whether the actor loss should update the critic or reward head. Is detaching an advantage the same as ensuring it was computed before the sampled action? Why not?

The diagram is schematic. Its tensor shapes describe the actual separate causal models; it is not measured attention or a shared-backbone architecture.

In [ ]:
fig,ax=plt.subplots(figsize=(12,5.2));ax.set(xlim=(0,12),ylim=(0,5));ax.axis("off")
boxes=[
 (".7","3.6","Prompt + generated prefix\ninteger IDs [B,T]"),
 ("4.1","3.6","Causal actor\nhidden [B,T,16] → logits [B,V]"),
 (".7","1.8","Separate causal critic\nhidden [B,T,16] → value [B]"),
 ("4.1","1.8","Rewards + EOS/cap masks\nTD → GAE → detached targets"),
 ("8.1","3.6","Sample token\nappend or emit EOS"),
 ("8.1","1.8","Saved character text RM\nfrozen scalar proxy"),
 ("4.1",".3","Independent quality rule\nreport only; no training gradient")]
for xs,ys,label in boxes:
    x,y=float(xs),float(ys)
    ax.add_patch(FancyBboxPatch((x,y),3,.9,boxstyle="round,pad=.04",
                              facecolor="#eef2f5",edgecolor="#617488"))
    ax.text(x+1.5,y+.45,label,ha="center",va="center",fontsize=10)
for start,end,label in [((3.75,4.05),(4.0,4.05),""),((7.15,4.05),(8.0,4.05),""),
                         ((3.75,2.25),(4.0,2.25),""),((9.6,3.5),(9.6,2.8),"text"),
                         ((8,2.25),(7.15,2.25),"reward"),
                         ((5.6,2.8),(5.6,3.5),"advantage")]:
    ax.annotate("",xy=end,xytext=start,arrowprops={"arrowstyle":"->","color":"#52616e"})
    if label:ax.text((start[0]+end[0])/2+.12,(start[1]+end[1])/2,label,fontsize=9)
ax.set_title("Structure only: actor and critic train; text reward stays frozen; quality is held aside")
plt.tight_layout();plt.show()


**Reference:** actor gradients reach current likelihoods; critic MSE reaches its own causal features and value head. Advantages, critic targets and old likelihoods are detached. The reward is frozen. Statistical action independence is a separate condition: stopping a gradient cannot repair a baseline that already used the chosen action.

![Saved actor critic reward schematic](../figures/chapter-12/day-20-04_learned_critics_and_frozen_text_rewards-01.png)

## Exercise 2 — did the trajectory end, or did observation end?

One row earns terminal reward 2 at EOS. The other is observed for two zero-reward actions, then capped with continuation value 1.2. Predict its lambda-one targets at gamma 0.9. What happens if we falsely mark that cap terminal?

In [ ]:
rewards=torch.tensor([[0.,0.,2.],[0.,0.,0.]],dtype=torch.float64)
values=torch.tensor([[.5,.7,.9],[.1,.2,.3]],dtype=torch.float64)
mask=torch.tensor([[True,True,True],[True,True,False]])
terminal=torch.tensor([[False,False,True],[False,False,False]])
bootstrap=torch.tensor([9.,1.2],dtype=torch.float64)
correct=gae_targets(rewards,values,mask,terminal,bootstrap,gamma=.9,lam=1)
broken_terminal=terminal.clone();broken_terminal[1,1]=True
broken=gae_targets(rewards,values,mask,broken_terminal,bootstrap,gamma=.9,lam=1)
print("Correct lambda-one targets:",correct["targets"])
print("Broken cap-as-EOS targets:",broken["targets"])
torch.testing.assert_close(correct["targets"][1,:2],torch.tensor([.972,1.08],dtype=torch.float64))
fig,(heat,curve)=plt.subplots(1,2,figsize=(10.5,4))
matrix=np.vstack([mask.numpy(),terminal.numpy(),correct["td"].numpy(),correct["advantages"].numpy()])
im=heat.imshow(matrix,cmap="coolwarm",vmin=-1.2,vmax=2,aspect="auto")
heat.set(xticks=range(3),xticklabels=["action 1","action 2","action 3"],
         yticks=range(8),yticklabels=["mask: EOS","mask: cap","terminal: EOS","terminal: cap",
                                     "TD: EOS","TD: cap","GAE: EOS","GAE: cap"],
         title="Actual tensors: padding is masked")
for i in range(8):
    for j in range(3):heat.text(j,i,f"{matrix[i,j]:.2f}",ha="center",va="center",fontsize=8)
grid=np.linspace(0,1,21)
for row,label,color in ((0,"EOS reward observed","#356a9b"),(1,"cap bootstrap kept","#b97139")):
    ys=[float(gae_targets(rewards,values,mask,terminal,bootstrap,gamma=.9,lam=float(l))["targets"][row,0]) for l in grid]
    curve.plot(grid,ys,color=color,label=label)
curve.set(xlabel="lambda",ylabel="first-state critic target",title="Lambda changes the target, not the event")
curve.legend(fontsize=9);plt.tight_layout();plt.show()


**Reference:** the capped targets are 0.972 and 1.08; falsely marking EOS sets both to zero at lambda one. The value nine after actual EOS is ignored. Lambda zero gives one-step TD; lambda one telescopes into observed rewards plus a nonterminal bootstrap. No future residual is invented beyond the prefix.

![Saved mask TD and lambda targets](../figures/chapter-12/day-20-04_learned_critics_and_frozen_text_rewards-02.png)

Change the bootstrap, then change only the terminal mask. Explain why these are different interventions.

## Exercise 3 — run a genuine two-update actor and learned critic

Predict which parameter families will receive gradients. This reference reloads an actual fitted reward with its expected external file identity. It is a separate two-update mechanism check, not the forty-update experiment or a selected favorable checkpoint.

In [ ]:
fit=result["reward_fits"]["balanced"]
frozen=load_char_reward(root/fit["export"]["path"],expected_file_sha256=fit["export"]["file_sha256"])
bundle={"frozen":frozen,"table":{(r["item_id"],tuple(r["tokens"])):{"raw":r["raw"],"proxy":r["proxy"]}
                                for r in fit["actual_reward_rows"]},
        "center":fit["center"],"scale":fit["scale"]}
before=state_hash(frozen.model)
small=run_arm(protocol,bundle,2001,"learned",updates=2)
print("Actor state changed:",small["actor_initial_sha256"]!=small["actor_final_sha256"])
print("Critic state changed:",small["critic_initial_sha256"]!=small["critic_final_sha256"])
print("Actual actor/critic norms:",[(h["actor_gradient"],h["critic_gradient"]) for h in small["history"]])
assert state_hash(frozen.model)==before
assert all(not p.requires_grad and p.grad is None for p in frozen.model.parameters())
first=small["training_ledger"][0]
print("First raw response IDs:",first["tokens"][0],"mask:",first["mask"][0],
      "values:",first["values"][0],"bootstrap:",first["bootstrap"][0],
      "advantages:",first["advantages"][0])


**Reference:** both causal trainable models change, but the reward stays byte-identical and has no gradients. Actor advantages and critic targets are detached. Our backbones are separate; shared features would deliberately receive both losses. A completed update proves a mechanism—not learned color matching or pretrained PPO quality.

## Exercise 4 — did mean proxy improvement buy independent quality?

Before revealing the curves, predict whether oracle values can rescue reward extrapolation. Every arm uses the same three actor seeds and sampled-path budget. The reward model was fitted on styled comparisons, while the actor may emit bare colors or repeat style words.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12.8,4),sharex=True)
arms=[("balanced","oracle"),("balanced","learned"),("balanced","noisy"),("confounded","learned")]
for arm,kind in arms:
    subset=[r for r in runs if r["reward_arm"]==arm and r["critic"]==kind]
    color=colors[kind] if arm=="balanced" else colors["confounded"]
    x=[p["update"] for p in subset[0]["panels"]]
    ys=[[],[],[]]
    for r in subset:
        for j,(split,field) in enumerate((("train","expected_terminal_proxy"),("train","expected_delivered_quality"),("test","expected_delivered_quality"))):
            ys[j].append([np.mean([z[field] for z in p["exact_finite_diagnostics"] if z["split"]==split]) for p in r["panels"]])
    for ax,vectors in zip(axes,ys):
        array=np.asarray(vectors);ax.plot(x,array.mean(0),color=color,label=f"{arm}/{kind}")
        ax.fill_between(x,array.min(0),array.max(0),color=color,alpha=.14)
for ax,title in zip(axes,("Eventual terminal proxy: train","Delivered quality: train cap3","Delivered quality: test cap3")):
    ax.set(xlabel="scheduled actor update",title=title)
axes[0].set(ylabel="exact finite expectation");axes[1].set(ylim=(0,1));axes[2].set(ylim=(0,1))
axes[0].legend(fontsize=8);plt.tight_layout();plt.show()


**Reference:** means and min–max envelopes retain all three seeds, with no best-checkpoint selection. These are exact finite-policy expectations, independently enumerable over fourteen paths, not measured language benchmark estimates. Eventual proxy beyond cap3 is distinct from delivered quality.

![Saved proxy and independent quality trajectories](../figures/chapter-12/day-20-04_learned_critics_and_frozen_text_rewards-03.png)

A critic can bootstrap an eventual proxy while the delivered prefix remains unfinished. Inspect the actual responses before attributing a unique cause.

## Exercise 5 — keep every failure in the denominator

Predict whether adding the fourth-position forced EOS repairs an answer with two repeated style words. Compare the actual final greedy scores across training, calibration, held-out test and controls. There are only two prompts per withheld slice; these values do not establish broad transfer.

In [ ]:
splits=["train","calibration","test","control"]
quality=np.array([[r["panels"][-1]["summaries"][f"{s}/greedy"]["quality"] for s in splits] for r in runs])
terminated=np.array([[r["panels"][-1]["summaries"][f"{s}/greedy"]["termination"] for s in splits] for r in runs])
labels=[f'{r["reward_arm"]}/{r["critic"]} {r["seed"]}' for r in runs]
fig,axes=plt.subplots(1,2,figsize=(12,6.4),sharey=True)
for ax,matrix,title in zip(axes,(quality,terminated),("Actual delivered greedy quality","Actual greedy EOS within cap3")):
    im=ax.imshow(matrix,vmin=0,vmax=1,cmap="Blues",aspect="auto")
    ax.set(xticks=range(4),xticklabels=splits,yticks=range(12),yticklabels=labels,title=title)
    for i in range(12):
        for j in range(4):ax.text(j,i,f"{matrix[i,j]:.2f}",ha="center",va="center",color="white" if matrix[i,j]>.6 else "black",fontsize=9)
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.65,label="fraction; all prompts retained")
plt.show()
for r in runs:
    failed=[(row["item_id"],row["text"],row["stop"]) for row in r["panels"][-1]["rows"]
            if row["mode"]=="greedy" and not row["quality"]]
    print(r["reward_arm"],r["critic"],r["seed"],"every greedy failure:",failed)
print("Actual budget:",sum(r["sampled_paths"] for r in runs),"paths;",
      sum(r["generated_response_tokens"] for r in runs),"valid response tokens;",
      sum(r["actor_updates"] for r in runs),"actor steps;",
      sum(r["critic_updates"] for r in runs),"critic steps")


**Reference:** complete EOS is necessary but not sufficient. Constant blue receives 0.5 quality on balanced two-color sources; repeated style fails even when the cap-four syntax supplies EOS. Learned-critic failures and sampled variability remain visible. Equal paths do not mean equal optimizer/enumeration cost.

![Saved all-seed quality and termination matrix](../figures/chapter-12/day-20-04_learned_critics_and_frozen_text_rewards-04.png)

## Exercise 6 — what did the reward never learn from pairs?

Inspect raw saved-model proxy scores for bare red/blue and styled variants on the original red-cup training source. Predict whether the balanced reward necessarily treats bare colors like its styled training responses. No policy gold label is used to select an answer here; this is a diagnostic.

In [ ]:
item=protocol["items"][0]
texts=["red","blue","red plain","blue plain","red fancy","blue fancy"]
fig,ax=plt.subplots(figsize=(10,4));x=np.arange(len(texts))
for offset,arm,color in ((-.17,"balanced","#356a9b"),(.17,"confounded","#775c91")):
    fit=result["reward_fits"][arm]
    saved=load_char_reward(root/fit["export"]["path"],expected_file_sha256=fit["export"]["file_sha256"])
    actual=saved.score_many([(item["prompt"],text) for text in texts]).numpy()
    scaled=np.tanh((actual-fit["center"])/fit["scale"])
    ax.bar(x+offset,scaled,width=.32,color=color,label=arm)
ax.set(xticks=x,xticklabels=texts,ylabel="actual saved text reward; train-only tanh scale",
       ylim=(-1.05,1.05),title="Bare responses lie outside styled preference comparisons")
ax.axhline(0,color="#888",linewidth=.8);ax.legend();plt.tight_layout();plt.show()


**Reference:** small pair loss does not constrain bare or repeated-style outputs. Prefix proxy scores are diagnostic, not earned rewards at emitted EOS. The finite actor and authored labels do not prove language competence, human alignment, critic superiority or a full PPO implementation. Any repair requires a new frozen recipe, not tuning these final-test rows.

![Saved actual balanced and confounded reward extrapolation](../figures/chapter-12/day-20-04_learned_critics_and_frozen_text_rewards-05.png)

The source, tests, complete raw training/evaluation records and frozen reward states are linked from the report. Animation production is not part of this notebook.